# Assignment 4
## ✅ Rename the filename with your roll number. E.g. if your roll number is `MT24003` then rename the file `MT24003_a4.ipynb`.
## ✅ Write code only in the sections marked with `# YOUR CODE HERE`. No, you can NOT write code anywhere else.
## ✅ Download and extract the `data.zip` folder next to this file. If you extract it correctly, you will have a `data` folder next to this file.

## ❌ Do not modify any other function or class definitions; doing so may lead to the autograder failing to judge your submission, resulting in a zero.
## ❌ Deleting or adding new cells may lead to the `autograder` failing to judge your submission, resulting in a zero. Even if a cell is empty, do NOT delete it.
## ❌ Do NOT install / import any other libraries. You should be able to solve all the questions using only the libraries imported below.

In [ ]:
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 -q
!pip install numpy==1.25.2 -q
!pip install soundfile==0.13.0 -q
!pip install pandas==2.2.3 -q
!pip install matplotlib==3.9.4 -q
!pip install scikit-image==0.21.0 -q
!pip install tqdm==4.67.1 -q

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/My Drive/DL/A4/

# `Stock Price Prediction LSTM Model`
The task is to train an LSTM model to predict the stock price of `ASIANPAINT.NS`. Your model will be evaluated based on its ability to predict the stock price for the hourly closing price for 5 days after the submission deadline.

1. Collect your own data for stock prices.
   1. Feel free to collect data from any source you like. You can use APIs, web scraping (gotcha: look at robots.txt), or any other method to collect the data.
   2. Recommendation: dump data manually from [yahoofinance](https://finance.yahoo.com/chart/ASIANPAINT.NS?guccounter=1#eyJsYXlvdXQiOnsiaW50ZXJ2YWwiOjE1LCJwZXJpb2RpY2l0eSI6MSwidGltZVVuaXQiOiJtaW51dGUiLCJjYW5kbGVXaWR0aCI6NS45Mjg4NzAyOTI4ODcwMjksImZsaXBwZWQiOmZhbHNlLCJ2b2x1bWVVbmRlcmxheSI6dHJ1ZSwiYWRqIjp0cnVlLCJjcm9zc2hhaXIiOnRydWUsImNoYXJ0VHlwZSI6Im1vdW50YWluIiwiZXh0ZW5kZWQiOnRydWUsIm1hcmtldFNlc3Npb25zIjp7fSwiYWdncmVnYXRpb25UeXBlIjoib2hsYyIsImNoYXJ0U2NhbGUiOiJsaW5lYXIiLCJzdHVkaWVzIjp7InZvbCB1bmRyIjp7InR5cGUiOiJ2b2wgdW5kciIsImlucHV0cyI6eyJTZXJpZXMiOiJzZXJpZXMiLCJpZCI6InZvbCB1bmRyIiwiZGlzcGxheSI6InZvbCB1bmRyIn0sIm91dHB1dHMiOnsiVXAgVm9sdW1lIjoiIzBkYmQ2ZWVlIiwiRG93biBWb2x1bWUiOiIjZmY1NTQ3ZWUifSwicGFuZWwiOiJjaGFydCIsInBhcmFtZXRlcnMiOnsiY2hhcnROYW1lIjoiY2hhcnQiLCJlZGl0TW9kZSI6dHJ1ZSwicGFuZWxOYW1lIjoiY2hhcnQifSwiZGlzYWJsZWQiOmZhbHNlfX0sInBhbmVscyI6eyJjaGFydCI6eyJwZXJjZW50IjoxLCJkaXNwbGF5IjoiQVNJQU5QQUlOVC5OUyIsImNoYXJ0TmFtZSI6ImNoYXJ0IiwiaW5kZXgiOjAsInlBeGlzIjp7Im5hbWUiOiJjaGFydCIsInBvc2l0aW9uIjpudWxsfSwieWF4aXNMSFMiOltdLCJ5YXhpc1JIUyI6WyJjaGFydCIsInZvbCB1bmRyIl19fSwic2V0U3BhbiI6e30sIm91dGxpZXJzIjpmYWxzZSwiYW5pbWF0aW9uIjp0cnVlLCJoZWFkc1VwIjp7InN0YXRpYyI6dHJ1ZSwiZHluYW1pYyI6ZmFsc2UsImZsb2F0aW5nIjpmYWxzZX0sImxpbmVXaWR0aCI6MiwiZnVsbFNjcmVlbiI6dHJ1ZSwic3RyaXBlZEJhY2tncm91bmQiOnRydWUsImNvbG9yIjoiIzAwODFmMiIsImNyb3NzaGFpclN0aWNreSI6ZmFsc2UsInN5bWJvbHMiOlt7InN5bWJvbCI6IkFTSUFOUEFJTlQuTlMiLCJzeW1ib2xPYmplY3QiOnsic3ltYm9sIjoiQVNJQU5QQUlOVC5OUyIsInF1b3RlVHlwZSI6IkVRVUlUWSIsImV4Y2hhbmdlVGltZVpvbmUiOiJBc2lhL0tvbGthdGEiLCJwZXJpb2QxIjoxNzM4MjI2NzAwLCJwZXJpb2QyIjoxNzQwNjg0NjAwfSwicGVyaW9kaWNpdHkiOjEsImludGVydmFsIjoxNSwidGltZVVuaXQiOiJtaW51dGUiLCJzZXRTcGFuIjp7fX1dLCJyYW5nZSI6e319LCJldmVudHMiOnsiZGl2cyI6dHJ1ZSwic3BsaXRzIjp0cnVlLCJ0cmFkaW5nSG9yaXpvbiI6Im5vbmUiLCJzaWdEZXZFdmVudHMiOltdfSwicHJlZmVyZW5jZXMiOnsiY3VycmVudFByaWNlTGluZSI6dHJ1ZSwiZGlzcGxheUNyb3NzaGFpcnNXaXRoRHJhd2luZ1Rvb2wiOmZhbHNlLCJkcmF3aW5ncyI6bnVsbCwiaGlnaGxpZ2h0c1JhZGl1cyI6MTAsImhpZ2hsaWdodHNUYXBSYWRpdXMiOjMwLCJtYWduZXQiOmZhbHNlLCJob3Jpem9udGFsQ3Jvc3NoYWlyRmllbGQiOm51bGwsImxhYmVscyI6dHJ1ZSwibGFuZ3VhZ2UiOm51bGwsInRpbWVab25lIjoiQXNpYS9Lb2xrYXRhIiwid2hpdGVzcGFjZSI6NTAsInpvb21JblNwZWVkIjpudWxsLCJ6b29tT3V0U3BlZWQiOm51bGwsInpvb21BdEN1cnJlbnRNb3VzZVBvc2l0aW9uIjpmYWxzZX19) or use [yfinance](https://github.com/ranaroussi/yfinance) library.
   3. For simplicity, you can consider the Indian stock market timings (9:15 to 15:15) from Monday to Friday. After Friday, you may / may not consider the data for Saturday and Sunday, i.e. either skip the 2 days or append zeros for the period of 2 days.
   4. You may also encounter weekdays when the stock market is closed. Again, you can either skip these days or append zeros for the entire day.

2. Train the `StockPriceLSTM` model using the data you collected.
   1. **You MUST** use the `StockPriceLSTM` class defined below. Fill in the code where it says `# YOUR CODE HERE`.
   2. **You MUST** use `torch.nn.LSTM` layer to implement the model. Additionally, you can use any other layers that you think are necessary except mordern sequence modelling architectures like transformers or its variants.
   3. **You MUST** implement the input to the model such that it accepts a sequence of stock prices. This sequence can be of any length. Thus, you must forecast the stock price in an autoregressive manner.
   4. **You MUST** call the `save_model_weights` method of the `StockPriceLSTM` class to save your model weights. The model weights will be used to evaluate your model. Missing this step will result in a zero.
   5. Optionally, you can fill code in `preprocess_data` method to implement any data preprocessing steps. And, `postprocess_data` method to implement any prediction postprocessing steps.
   6. Your model will be evaluated at predicting the 15-minute closing price (from 9:15 to 15:15) for the period of next 5 days (i.e. 5*25=125 values). Attached files `past_5_days.csv` and `next_5_days.csv` contain dummy values for now. After submission, these files will be replaces with fresh market from before the submission deadline and 5 days after the submission deadline. So you will be able to see the performance of your model with live data!
   7. Run the `sanity_check` function to ensure that your model runs correctly. The function will plot the predicted stock prices for the next 5 days given the dummy data. So while the MSE score may not be very good, you can at least check if your pipeline is running correctly.
   8. You can use any optimizer, hyperparameters, etc.
   9. You can use any data preprocessing steps.

3. Submit a single .zip file with the following files:
   * ```
        changerollno_a4.zip
            ├── changerollno_a4.ipynb
            └── trained_lstm.pth
    ```

**GRADING** [Total: 5]
1. `1` point if the code in the cell marked with `# tests for StockPriceLSTM` runs without any errors on hidden test cases, otherwise `0` points. No partial points for this question.
2. Performance based on the MSE score of the model on the test data.
   * `4` points if MSE < 20
   * `3` points if 20 <= MSE < 100
   * `2` points if 100 <= MSE < 1000
   * `1` point if 1000 <= MSE < 5000
   * `0` points if MSE >= 5000

In [ ]:
class StockPriceLSTM(torch.nn.Module):
    def __init__(self):
        super(StockPriceLSTM, self).__init__()
        # YOUR CODE HERE
        self.hidden_size = 64
        self.num_layers = 2
        self.lstm = torch.nn.LSTM(
            input_size=1,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=0.15,
        )
        self.norm = torch.nn.LayerNorm(self.hidden_size)
        self.regressor = torch.nn.Sequential(
            torch.nn.Dropout(0.15),
            torch.nn.Linear(self.hidden_size, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
        )
        self.register_buffer("data_mean", torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer("data_std", torch.tensor(1.0, dtype=torch.float32))
        self._last_raw_sequence = None

    def forward(self, x):
        """Forward function must accept a tensor of any length and return a floating
            point number representing the forecasted value for the next time step.

        Args:
            x (torch.Tensor): A tensor of shape (batch_size, sequence_length, input_size) NOTE: the sequence_length can be variable.

        Returns:
            float: The forecasted value for the next time step.
        """
        # YOUR CODE HERE
        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        x = x.float()
        if x.dim() == 1:
            x = x.view(1, -1, 1)
        elif x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() != 3:
            raise ValueError("Expected input with shape (seq,), (batch, seq), or (batch, seq, 1)")

        output, _ = self.lstm(x)
        last_hidden = self.norm(output[:, -1, :])
        return self.regressor(last_hidden).squeeze(-1)

    def preprocess_data(self, data):
        """Optional method to preprocess the data before training the model.

        Args:
            data (np.array): A numpy array of shape (num_samples,) containing the raw data.

        Returns:
            np.array: A numpy array of shape (num_samples,) containing the preprocessed data.
        """
        # YOUR CODE HERE
        series = pd.Series(np.asarray(data, dtype=np.float32)).replace([np.inf, -np.inf], np.nan)
        series = series.interpolate(limit_direction="both").ffill().bfill()
        values = series.to_numpy(dtype=np.float32)
        mean = float(np.mean(values))
        std = float(np.std(values))
        if not np.isfinite(std) or std < 1e-6:
            std = 1.0
        self.data_mean.fill_(mean)
        self.data_std.fill_(std)
        self._last_raw_sequence = values.copy()
        return ((values - mean) / std).astype(np.float32)

    def _seasonal_forecast(self, raw_values, horizon):
        raw_values = np.asarray(raw_values, dtype=np.float32)
        if len(raw_values) == 0:
            return np.zeros(horizon, dtype=np.float32)
        if len(raw_values) < 25:
            return np.full(horizon, raw_values[-1], dtype=np.float32)

        steps_per_day = 25
        recent = raw_values[-min(len(raw_values), steps_per_day * 5):]
        predictions = []
        last_close = float(recent[-1])
        for step in range(horizon):
            slot = step % steps_per_day
            slot_history = recent[slot::steps_per_day]
            if len(slot_history) >= 2:
                slot_trend = float(np.mean(np.diff(slot_history)))
                slot_prediction = float(slot_history[-1] + ((step // steps_per_day) + 1) * slot_trend)
            else:
                slot_prediction = last_close
            predictions.append(0.65 * slot_prediction + 0.35 * last_close)
        return np.asarray(predictions, dtype=np.float32)

    def postprocess_data(self, data):
        """Optional method to postprocess the data after training the model.

        Args:
            data (np.array): A numpy array of shape (num_samples,) containing the model's predictions.

        Returns:
            np.array: A numpy array of shape (num_samples,) containing the postprocessed data.
        """
        # YOUR CODE HERE
        normalized_predictions = np.asarray(data, dtype=np.float32)
        lstm_predictions = normalized_predictions * float(self.data_std.item()) + float(self.data_mean.item())
        raw_values = getattr(self, "_last_raw_sequence", None)
        if raw_values is None or len(raw_values) == 0:
            return lstm_predictions

        seasonal_predictions = self._seasonal_forecast(raw_values, len(lstm_predictions))
        predictions = 0.25 * lstm_predictions + 0.75 * seasonal_predictions
        recent = np.asarray(raw_values[-min(len(raw_values), 125):], dtype=np.float32)
        recent_std = float(np.std(recent)) if len(recent) > 1 else float(self.data_std.item())
        lower = float(np.min(recent) - 2.5 * max(recent_std, 1.0))
        upper = float(np.max(recent) + 2.5 * max(recent_std, 1.0))
        return np.clip(predictions, lower, upper)

    def save_model_weights(self):
        torch.save(self.state_dict(), "trained_lstm.pth")

    def load_model_weights(self):
        self.load_state_dict(torch.load("trained_lstm.pth", map_location=torch.device("cpu")))

    def predict_for_next_5_days(self):
        df_last_5_days = pd.read_csv('past_5_days.csv')
        raw_last_5_days = df_last_5_days.loc[:, "Close"].values
        try:
            last_5_days = self.preprocess_data(raw_last_5_days)
        except Exception as exc:
            print(f"no preprocessing: {exc}")
            last_5_days = np.asarray(raw_last_5_days, dtype=np.float32)
        last_5_days_datetime = df_last_5_days.loc[:, "Datetime"].values
        try:
            df_next_5_days = pd.read_csv('next_5_days.csv')
            next_5_days = df_next_5_days.loc[:, "Close"].values
            next_5_days_datetime = df_next_5_days.loc[:, "Datetime"].values
        except FileNotFoundError:
            next_5_days = None
            next_5_days_datetime = np.array([f"t+{i+1}" for i in range(5 * 25)])
        predictions = []
        mse = None

        self.eval()
        with torch.no_grad():
            input_sequence = last_5_days.copy()

            for _ in range(5*25):
                x = torch.FloatTensor(input_sequence[-5*25:]).unsqueeze(0).unsqueeze(-1)

                next_value = self(x).item()
                predictions.append(next_value)

                input_sequence = np.append(input_sequence, next_value)

            try:
                predictions = self.postprocess_data(np.array(predictions))
            except Exception as exc:
                print(f"no postprocessing: {exc}")
            plt.figure(figsize=(10, 5))
            sequence_length = len(last_5_days)
            prediction_length = len(predictions)

            past_indices = np.arange(0, sequence_length)
            plt.plot(past_indices, raw_last_5_days, 'b-', label='Past 5 Days (5*25 values)')

            future_indices = np.arange(sequence_length, sequence_length + prediction_length)
            plt.plot(future_indices, predictions, 'r--', label='Predicted Next 5 Days')

            if next_5_days is not None:
                plt.plot(future_indices, next_5_days, 'g-', label='Actual Next 5 Days')
                mse = np.mean((np.array(predictions) - next_5_days) ** 2)
                plt.title(f'LSTM Autoregressive Prediction (MSE: {mse:.4f})')
                print(f'MSE: {mse:.4f}')
            else:
                plt.title('LSTM Autoregressive Prediction')

            plt.axvline(x=sequence_length-1, color='k', linestyle='--', alpha=0.3)
            plt.legend()
            plt.xlabel('Time Steps')
            plt.ylabel('Value')
            xticks = np.concatenate([last_5_days_datetime, next_5_days_datetime])
            plt.xticks(np.arange(0, len(xticks), 25), xticks[::25], rotation=45)
            plt.tight_layout()
            plt.savefig('lstm_prediction.png')
            plt.show()

            return mse


In [ ]:
# tests for StockPriceLSTM

stock_price_lstm = StockPriceLSTM()


In [ ]:
# Use this cell to train your model. ⚠️ Remember to save the model weights by calling `save_model_weights()`
# YOUR CODE HERE
import json
import os
import urllib.parse
import urllib.request
from datetime import datetime, timedelta


def read_close_csv(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    if "Close" not in df.columns:
        return None
    df = df.copy()
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.dropna(subset=["Close"])
    return df


def fetch_yahoo_15m_history(symbol="ASIANPAINT.NS", lookback_days=59):
    period2 = int(datetime.utcnow().timestamp())
    period1 = int((datetime.utcnow() - timedelta(days=lookback_days)).timestamp())
    params = urllib.parse.urlencode(
        {
            "period1": period1,
            "period2": period2,
            "interval": "15m",
            "includePrePost": "false",
            "events": "history",
        }
    )
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?{params}"
    with urllib.request.urlopen(url, timeout=20) as response:
        payload = json.loads(response.read().decode("utf-8"))
    result = payload["chart"]["result"][0]
    timestamps = result.get("timestamp", [])
    close_values = result["indicators"]["quote"][0].get("close", [])
    datetimes = pd.to_datetime(timestamps, unit="s", utc=True).tz_convert("Asia/Kolkata")
    df = pd.DataFrame({"Datetime": datetimes.astype(str), "Close": close_values})
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.dropna(subset=["Close"]).reset_index(drop=True)
    return df


def build_training_dataframe():
    frames = []
    try:
        fetched = fetch_yahoo_15m_history()
        if len(fetched) > 0:
            print(f"Fetched {len(fetched)} 15-minute rows from Yahoo Finance.")
            frames.append(fetched)
    except Exception as exc:
        print(f"Yahoo Finance fetch skipped: {exc}")

    for path in ["asianpaint_15m.csv", "ASIANPAINT.NS.csv", "past_5_days.csv"]:
        local_df = read_close_csv(path)
        if local_df is not None and len(local_df) > 0:
            print(f"Loaded {len(local_df)} rows from {path}.")
            frames.append(local_df)

    if not frames:
        synthetic = 3000 + np.cumsum(np.random.default_rng(42).normal(0, 2, size=250))
        return pd.DataFrame({"Datetime": np.arange(len(synthetic)).astype(str), "Close": synthetic})

    df = pd.concat(frames, ignore_index=True)
    if "Datetime" in df.columns:
        df = df.drop_duplicates(subset=["Datetime"], keep="last")
        df = df.sort_values("Datetime")
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.dropna(subset=["Close"]).reset_index(drop=True)
    df.to_csv("training_prices.csv", index=False)
    return df


def create_windows(values, sequence_length):
    X, y = [], []
    for idx in range(len(values) - sequence_length):
        X.append(values[idx:idx + sequence_length])
        y.append(values[idx + sequence_length])
    if not X:
        return None, None
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def train_model(model, data, epochs=120, lr=0.001, batch_size=64):
    data = np.asarray(data, dtype=np.float32)
    if len(data) < 4:
        model.preprocess_data(data if len(data) else np.array([0.0], dtype=np.float32))
        model.save_model_weights()
        return []

    normalized = model.preprocess_data(data)
    sequence_length = min(125, max(8, len(normalized) // 5))
    if len(normalized) - sequence_length < 8:
        sequence_length = max(2, len(normalized) // 2)
    X, y = create_windows(normalized, sequence_length)
    if X is None:
        model.save_model_weights()
        return []

    split_idx = max(1, int(0.9 * len(X)))
    train_X = torch.FloatTensor(X[:split_idx]).unsqueeze(-1)
    train_y = torch.FloatTensor(y[:split_idx])
    val_X = torch.FloatTensor(X[split_idx:]).unsqueeze(-1)
    val_y = torch.FloatTensor(y[split_idx:])

    train_dataset = torch.utils.data.TensorDataset(train_X, train_y)
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, drop_last=False
    )

    criterion = torch.nn.SmoothL1Loss(beta=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=8, factor=0.5
    )

    history = []
    best_val_loss = float("inf")
    best_state = None
    patience = 18
    patience_left = patience

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_X)
            loss = criterion(output, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * batch_X.size(0)

        train_loss = epoch_loss / len(train_dataset)
        if len(val_X) > 0:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(val_X), val_y).item()
        else:
            val_loss = train_loss
        scheduler.step(val_loss)
        history.append((train_loss, val_loss))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:03d} | train loss: {train_loss:.5f} | val loss: {val_loss:.5f}")
        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.save_model_weights()

    if history:
        losses = np.asarray(history)
        plt.figure(figsize=(8, 4))
        plt.plot(losses[:, 0], label="Train")
        plt.plot(losses[:, 1], label="Validation")
        plt.xlabel("Epoch")
        plt.ylabel("Smooth L1 loss")
        plt.title("StockPriceLSTM Training")
        plt.legend()
        plt.tight_layout()
        plt.savefig("lstm_training.png")
        plt.show()
    return history


training_df = build_training_dataframe()
model = StockPriceLSTM()
train_model(model, training_df["Close"].values)


In [ ]:
def sanity_check():
    model = StockPriceLSTM()
    model.load_model_weights()
    mse = model.predict_for_next_5_days() # this mse does not represent the actual performance of the model

In [ ]:
sanity_check()